# Create Survival Prediction Model using PyTorch

One strategy would be to replicate the [DeepSurv model](https://arxiv.org/abs/1606.00931), which is a deep neural network for survival analysis, essentially a nonlinear version of the Cox proportional hazard model. 

DeepSurv takes a list of features, and learns a risk function from input features. 
No restrictions, all types of data: clinical, gene expression, genetic mutations, are all treated the same. 

## Multi-modal Learning Model

Our data includes 3 unique modalities: 
- Clinical metadata
- Z-scored Gene Expression 
- Genetic Mutation data

So we will create separate encoders for each modality

Clinical features ──► Clinical MLP    ┐

Expression matrix ──► Expr Encoder ├─► Fusion ─► Risk score ─► Cox loss

Mutation matrix ────► Mut Encoder ─┘

## Architecture
1. Clinical Encoder (MLP): Low depth, minimal regularization
2. Gene Expression Encoder (Autoencoder or Bottleneck MLP)
Options: 
- Variance Filtered Genes
- Pathway Scores
- Autoencoder Latent Space
3. Mutational Autoencoder
- Binary gene-level mutation matrix
- Tumor mutational burden

## Training strategy 

1. Train clinical-only DeepSurv model
2. Add expression autoencoder (similar to Cox model) 
3. Add mutation autoencoder 
4. Fine-tune all layers

## Evaluation Strategies
Stratify patients by predicted risk tertiles

Plot Kaplan Meier curves

Compare against one another:

CoxPH (clinical)

CoxPH (clinical + expression)

DeepSurv multimodal

In [9]:
# load libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pyhere as here


### Load Data

In [12]:
### Load data 

# load clinical metadata
clinical = pd.read_csv(here.here("data", "processed","clinical_data.csv"))
# load expression data
expression_data = pd.read_csv(here.here("data", "processed","expression_data.csv"))
# mutation binary data
mutation_binary = pd.read_csv(here.here("data", "processed","mutation_binary_data.csv"))
# mutation classified data
mutation_classified = pd.read_csv(here.here("data", "processed","mutation_classified_data.csv"))

### Loss Function

In [14]:
# partial likelihood loss function for Cox Proportional Hazards model
def cox_ph_loss(risk_scores, times, events):
    """
    Negative partial log-likelihood for Cox PH
    """
    order = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events = events[order]

    log_cumsum = torch.logcumsumexp(risk_scores, dim=0)
    loss = -torch.sum((risk_scores - log_cumsum) * events)
    return loss / events.sum()

### Modality-specific Encoders

In [16]:
import torch.nn as nn

# clinical data encoder
class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [17]:
# Expression data encoder
class ExpressionEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [19]:
# Mutation data encoder
class MutationEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

### Multimodal DeepServ Model

In [22]:
# Multi-modal DeepSurv model, combining clinical, expression, and mutation data
class MultiModalDeepSurv(nn.Module):
    def __init__(self, clin_dim, expr_dim, mut_dim):
        super().__init__()

        self.clin_enc = ClinicalEncoder(clin_dim)
        self.expr_enc = ExpressionEncoder(expr_dim)
        self.mut_enc  = MutationEncoder(mut_dim)

        self.head = nn.Sequential(
            nn.Linear(32 + 128 + 64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x_clin, x_expr, x_mut):
        z_clin = self.clin_enc(x_clin)
        z_expr = self.expr_enc(x_expr)
        z_mut  = self.mut_enc(x_mut)

        z = torch.cat([z_clin, z_expr, z_mut], dim=1)
        return self.head(z).squeeze(-1)


# Training function for one epoch
def train_epoch(model, optimizer, x_clin, x_expr, x_mut, time, event):
    model.train()
    optimizer.zero_grad()

    risk = model(x_clin, x_expr, x_mut)
    loss = cox_ph_loss(risk, time, event)

    loss.backward()
    optimizer.step()
    return loss.item()


### Example run with randomized data

In [23]:
# example run with randomized data
N = 1500  # number of samples
C = 30      # clinical
G = 1000   # expression
M = 200    # mutation

x_clin = torch.randn(N, C)
x_expr = torch.randn(N, G)
x_mut  = torch.randint(0, 2, (N, M)).float()

time  = torch.rand(N) * 100
event = torch.randint(0, 2, (N,)).float()

model = MultiModalDeepSurv(C, G, M)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    loss = train_epoch(
        model,
        optimizer,
        x_clin,
        x_expr,
        x_mut,
        time,
        event
    )
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss:.4f}")


Epoch 0 | Loss: 6.3051
Epoch 10 | Loss: 5.7033
Epoch 20 | Loss: 4.6340
Epoch 30 | Loss: 4.5254
Epoch 40 | Loss: 4.3099


### Evaluation (C-index)

In [24]:
from lifelines.utils import concordance_index

model.eval()
with torch.no_grad():
    risk = model(x_clin, x_expr, x_mut).numpy()

c_index = concordance_index(
    time.numpy(),
    -risk,
    event.numpy()
)

print("C-index:", c_index)


C-index: 0.9827553716259907


Remarkably good C-index! (Almost like the data isn't real! )